# WM-811K — Defect-pattern CNN

A small convolutional classifier for the 9 defect classes (8 patterns + 'none'). Goals:

- Beat the trivial baseline (predict the majority class) by a clear margin on macro-F1.
- Show that the model picks up the spatially distinctive patterns (Center, Donut, Edge-Ring) cleanly, and identify where it gets confused.

Architecture is deliberately small — three conv blocks, global average pool, two dense layers. The point isn't a state-of-the-art number; it's a clean baseline I can reason about.

Implementation is in PyTorch. (I started with Keras but ran into a long TensorFlow install on this machine — for a small CNN like this, PyTorch is the lighter dependency anyway.)

**Prereq:** `data/raw/wm811k/LSWMD.pkl` must be present. See the EDA notebook for download instructions.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from src.data_io import load_wm811k

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42
IMG_SIZE = 64
BATCH = 128
EPOCHS = 12

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

## Load and prepare images

Resize every wafer map to 64×64 with nearest-neighbour interpolation (preserves the discrete 0/1/2 values), then normalize to floats in [0, 1] by dividing by 2. Encode labels as integers.

The full labeled set is ~172k wafer maps; resizing them all takes a couple of minutes.

In [ ]:
from PIL import Image

df = load_wm811k("../data/raw/wm811k/LSWMD.pkl", labeled_only=True)

def resize_one(arr, size=IMG_SIZE):
    img = Image.fromarray(arr.astype(np.uint8))
    img = img.resize((size, size), resample=Image.NEAREST)
    return np.asarray(img, dtype=np.float32) / 2.0

X = np.stack([resize_one(w) for w in df["waferMap"].values])[:, None, :, :]
classes = sorted(df["label"].unique().tolist())
label_to_idx = {c: i for i, c in enumerate(classes)}
y = df["label"].map(label_to_idx).values

print("X:", X.shape, "dtype", X.dtype)
print("classes:", classes)

## Stratified train / val / test split

70 / 15 / 15. Stratification matters here — the rare classes (Donut, Near-full, Scratch) have a few hundred examples each at most, and an unstratified split could easily under-represent them in any one fold.

In [ ]:
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15 / 0.85, stratify=y_tv, random_state=RANDOM_STATE
)
print("train", X_train.shape, "val", X_val.shape, "test", X_test.shape)

cw = compute_class_weight("balanced", classes=np.arange(len(classes)), y=y_train)
print("class weights:", {classes[i]: round(float(cw[i]), 2) for i in range(len(classes))})

def to_loader(Xa, ya, shuffle=False):
    ds = TensorDataset(torch.from_numpy(Xa), torch.from_numpy(ya).long())
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=0)

train_loader = to_loader(X_train, y_train, shuffle=True)
val_loader   = to_loader(X_val,   y_val)
test_loader  = to_loader(X_test,  y_test)

## Model

Three small conv blocks → global average pool → dense head. BatchNorm between conv and activation, dropout in the head.

In [ ]:
class WaferCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.Conv2d(oc, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.b1 = block(1, 32)
        self.b2 = block(32, 64)
        self.b3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = self.b1(x); x = self.b2(x); x = self.b3(x)
        return self.head(x)

model = WaferCNN(len(classes)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")

## Train

Class-weighted cross-entropy (otherwise the model just learns 'none'), Adam at 1e-3, LR halving on validation-loss plateau. I track val loss and val macro-F1 each epoch — F1 is what we actually care about, val loss is what the LR scheduler watches.

In [ ]:
weights = torch.tensor(cw, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_f1 = -1.0
best_state = None

def run_epoch(loader, train=False):
    model.train(train)
    total, count, preds, trues = 0.0, 0, [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * yb.size(0); count += yb.size(0)
        preds.append(logits.argmax(1).cpu().numpy()); trues.append(yb.cpu().numpy())
    return total / count, np.concatenate(preds), np.concatenate(trues)

for epoch in range(1, EPOCHS + 1):
    tr_loss, _, _ = run_epoch(train_loader, train=True)
    va_loss, va_pred, va_true = run_epoch(val_loader, train=False)
    va_f1 = f1_score(va_true, va_pred, average="macro")
    scheduler.step(va_loss)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss); history["val_f1"].append(va_f1)
    print(f"epoch {epoch:2d}  train_loss {tr_loss:.4f}  val_loss {va_loss:.4f}  val_macroF1 {va_f1:.3f}")
    if va_f1 > best_f1:
        best_f1 = va_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"\nbest val macro-F1: {best_f1:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history["val_f1"], label="val macro-F1", color="#c44e52")
axes[1].set_title("val macro-F1"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## Evaluation on the held-out test set

Macro-F1 is the headline metric — it weights every class equally, which is the right call given the imbalance. Per-class precision/recall/F1 in the classification report shows where the model actually works and where it struggles.

In [ ]:
_, y_pred, y_true = run_epoch(test_loader, train=False)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"macro F1 on test: {macro_f1:.3f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=ax, cbar=False)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Confusion matrix (row-normalized)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()

## Where it gets things wrong

Twelve random misclassified test wafers, with the predicted and true labels. Useful for spotting *systematic* failure modes — e.g. the network confusing 'Loc' (localized cluster of failures) with 'Scratch' (linear cluster) when the cluster happens to be roughly linear.

In [ ]:
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#ffffff", "#cccccc", "#c44e52"])

wrong = np.where(y_pred != y_true)[0]
rng = np.random.default_rng(0)
picks = rng.choice(wrong, size=min(12, len(wrong)), replace=False) if len(wrong) else np.array([], dtype=int)

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
for ax, idx in zip(axes.flat, picks):
    img = (X_test[idx, 0] * 2).astype(np.uint8)
    ax.imshow(img, cmap=cmap, vmin=0, vmax=2)
    ax.set_title(f"pred: {classes[y_pred[idx]]}\ntrue: {classes[y_true[idx]]}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)
torch.save(model.state_dict(), "../models/wm811k_cnn.pt")
pd.DataFrame({"macro_f1": [macro_f1]}).to_csv("../reports/wm811k_results.csv", index=False)
print("saved model and macro-F1 result")

## Honest notes

- The model gets most of its score from the spatially distinctive classes (Center, Donut, Edge-Ring, Near-full). Those have characteristic shapes that a small CNN can latch onto in a few epochs.
- The hardest classes are 'Loc', 'Scratch', and 'Random'. They overlap visually — a short scratch and a small localized cluster of failures look similar at 64×64. A higher input resolution, or rotation augmentation to teach orientation invariance for 'Scratch', would probably help.
- I didn't use data augmentation. Wafer maps have meaningful orientation (notches, flat edges) so naive rotation/flipping can change the meaning of a pattern; doing this properly requires domain-aware augmentation. Left as future work.
- The 'none' class is by far the largest and the network does well on it just from frequency. Class weights pull the gradient toward minority classes; without them macro-F1 collapses.